# **Deskripsi Dataset**

Dataset yang digunakan merupakan gambaran tentang mahasiswa yang terdaftar dalam 
berbagai program sarjana yang ditawarkan oleh perguruan tinggi. Data ini mencakup data 
demografis, faktor sosial-ekonomi, dan informasi kinerja akademik yang dapat digunakan 
untuk menganalisis faktor-faktor yang mungkin memprediksi tingkat kesuksesan akademik 
mahasiswa.

# **Requirements and Config**

In [30]:
# !pip install seaborn matplotlib numpy pandas

In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

sys.path.append(os.getcwd())
from allyouneed.tree import DecisionTreeClassifier
from allyouneed.model_selection import StratifiedKFold
from allyouneed.metrics import Accuracy, F1Score
from allyouneed.preprocessing import OneHotEncoder, LabelEncoder
from allyouneed.feature_selection import ForwardFeatureSelection, BackwardFeatureElimination

import warnings
warnings.filterwarnings("ignore")


pd.set_option('display.max_columns', None)

class settings:
    DATA_DIR = "../dataset/"
    TRAIN_FILE = "train.csv"
    TEST_FILE = "test.csv"
    SUBMISSION_FILE = "sample_submission.csv"
    INDEX_COL = "Student_ID"
    SEED = 42
    TARGET = "Target"
    TRAIN_PATH = DATA_DIR + TRAIN_FILE
    TEST_PATH = DATA_DIR + TEST_FILE
    SUBMISSION_PATH = DATA_DIR + SUBMISSION_FILE

In [32]:
train = pd.read_csv(settings.TRAIN_PATH, index_col=settings.INDEX_COL)
test = pd.read_csv(settings.TEST_PATH, index_col=settings.INDEX_COL)
submission = pd.read_csv(settings.SUBMISSION_PATH, index_col=settings.INDEX_COL)
train.select_dtypes(include='int64').describe()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Nacionality,Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,Displaced,Educational special needs,Debtor,Tuition fees up to date,Gender,Scholarship holder,Age at enrollment,International,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (without evaluations)
count,3096.000000,3096.000000,3096.000000,3096.000000,3096.00000,3096.000000,3096.000000,3096.000000,3096.000000,3096.000000,3096.000000,3096.000000,3096.000000,3096.000000,3096.000000,3096.000000,3096.000000,3096.000000,3096.000000,3096.000000,3096.000000,3096.000000,3096.000000,3096.000000,3096.000000,3096.000000,3096.000000,3096.000000,3096.000000
mean,1.166021,18.372416,1.717700,8832.330749,0.89438,4.672804,1.797158,19.361111,22.146641,10.536176,10.743863,0.548450,0.011951,0.110142,0.880814,0.354328,0.252584,23.216408,0.023256,0.713501,6.245801,8.242571,4.697674,0.140181,0.539729,6.201227,8.015181,4.410853,0.141150
std,0.573701,17.456612,1.292362,2116.515003,0.30740,10.387415,6.781964,15.568871,15.329250,25.028462,24.464090,0.497727,0.108683,0.313117,0.324060,0.478386,0.434565,7.614394,0.150739,2.365945,2.501305,4.186787,3.087995,0.672843,1.906634,2.223507,3.966593,3.022767,0.718398
min,1.000000,1.000000,0.000000,33.000000,0.00000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,17.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,1.000000,9085.000000,1.00000,1.000000,1.000000,2.000000,3.000000,4.000000,4.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,19.000000,0.000000,0.000000,5.000000,6.000000,3.000000,0.000000,0.000000,5.000000,6.000000,2.000000,0.000000
50%,1.000000,17.000000,1.000000,9238.000000,1.00000,1.000000,1.000000,19.000000,19.000000,5.000000,7.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,20.000000,0.000000,0.000000,6.000000,8.000000,5.000000,0.000000,0.000000,6.000000,8.000000,5.000000,0.000000
75%,1.000000,39.000000,2.000000,9556.000000,1.00000,1.000000,1.000000,37.000000,37.000000,9.000000,9.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,25.000000,0.000000,0.000000,7.000000,10.000000,6.000000,0.000000,0.000000,7.000000,10.000000,6.000000,0.000000
max,6.000000,57.000000,9.000000,9991.000000,1.00000,43.000000,108.000000,44.000000,44.000000,194.000000,195.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,70.000000,1.000000,20.000000,26.000000,45.000000,26.000000,12.000000,19.000000,23.000000,33.000000,20.000000,12.000000


# **Exploratory Data Analysis**

In [ ]:
# 1. Pilih kolom int64
int_cols = train.select_dtypes(include="int64").columns

# 2. Salinan agar tidak mengubah train asli
df_cat = train[int_cols].copy()

# 3. Convert semua kolom int ke categorical (object)
df_cat = df_cat.astype("object")

# 4. Hitung nunique dan mode
summary = []

for col in df_cat.columns:
    nunique = df_cat[col].nunique()
    mode = df_cat[col].mode().iloc[0] if nunique > 0 else None
    summary.append({
        "column": col,
        "nunique": nunique,
        "mode": mode
    })

summary_df = pd.DataFrame(summary)

summary_df.sort_values(by="nunique")


# **Preprocessing**

## **Data Cleaning**

In [34]:
def drop_columns(df):
    col = []
    return df.drop(columns=col)

## **Data Transformation**

In [35]:
label_encoder = LabelEncoder()
label_encoder.fit(train[settings.TARGET])

## **Feature Selection**

## **Dimensionality Reduction**

## **Pipeline**

In [36]:
class Preprocessing:
    def __init__(self, categorical_cols=None):
        self.categorical_cols = categorical_cols or []
        self.ohe = OneHotEncoder(columns=categorical_cols, dtype=float)
        self.numeric_cols = []
        self.fitted = False

    def fit(self, X):
        if not isinstance(X, pd.DataFrame):
            raise ValueError("X must be a pandas DataFrame")

        self.numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        self.numeric_cols = [col for col in self.numeric_cols if col not in self.categorical_cols]
        self.ohe.fit(X)
        self.fitted = True
        return self

    def transform(self, X):
        if not self.fitted:
            raise ValueError("Preprocessing must be fitted before transform()")

        if not isinstance(X, pd.DataFrame):
            raise ValueError("X must be a pandas DataFrame")

        # Numeric features (tetap DataFrame)
        X_num = X[self.numeric_cols].astype(float)

        # Categorical features (one-hot encoded)
        X_cat_array = self.ohe.transform(X)
        X_cat = pd.DataFrame(
            X_cat_array,
            columns=self.ohe.feature_names_,
            index=X.index
        )

        # Combine dan return DataFrame
        result = pd.concat([X_num, X_cat], axis=1)
        return result

    def fit_transform(self, X):
        return self.fit(X).transform(X)


CATEGORICAL_COLS = [
    "Marital status",
    "Course",
    "Previous qualification",
    "Application mode",
    "Nacionality",
    "Application order"
]

DROPPED_COLS = [
    
]

preprocessor = Preprocessing(categorical_cols=CATEGORICAL_COLS)
X_train, y_train = train.drop(columns=[settings.TARGET]), label_encoder.transform(train[settings.TARGET])
X_train = preprocessor.fit_transform(X_train)
y_train = label_encoder.transform(train[settings.TARGET])

ffs = ForwardFeatureSelection(
    estimator=DecisionTreeClassifier(random_state=settings.SEED),
    n_features_to_select=20,
    scoring=F1Score(average='macro'),
    verbose=True,
    feature_names=X_train.columns.tolist(),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=settings.SEED),
    n_jobs=8
)

In [ ]:
ffs.fit(X_train, y_train)
# ffs.selected_features_

Forward Feature Selection
Total features: 113
Target features: 20

Iteration 1/20
Testing 113 features...
Selected: Curricular units 2nd sem (approved) (score: 0.6134)
Current features: ['Curricular units 2nd sem (approved)']

Iteration 2/20
Testing 112 features...
Selected: Course__9773 (score: 0.6222)
Current features: ['Curricular units 2nd sem (approved)', 'Course__9773']

Iteration 3/20
Testing 111 features...
Selected: Curricular units 2nd sem (credited) (score: 0.6399)
Current features: ['Curricular units 2nd sem (approved)', 'Course__9773', 'Curricular units 2nd sem (credited)']

Iteration 4/20
Testing 110 features...
Selected: Debtor (score: 0.6473)
Current features: ['Curricular units 2nd sem (approved)', 'Course__9773', 'Curricular units 2nd sem (credited)', 'Debtor']

Iteration 5/20
Testing 109 features...
Selected: Tuition fees up to date (score: 0.6604)
Current features: ['Curricular units 2nd sem (approved)', 'Course__9773', 'Curricular units 2nd sem (credited)', 'Debtor

# **Modeling and Validation**

## **Metrics**

In [ ]:
# ============================================================
# Utility: Confusion Matrix
# ============================================================
class ConfusionMatrix:
    """
    Computes confusion matrix using NumPy.
    """

    def __call__(self, y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        labels = np.unique(np.concatenate([y_true, y_pred]))
        label_to_idx = {label: idx for idx, label in enumerate(labels)}
        
        cm = np.zeros((len(labels), len(labels)), dtype=int)

        for t, p in zip(y_true, y_pred):
            cm[label_to_idx[t], label_to_idx[p]] += 1
        
        return cm, labels

# ============================================================
# Metric Collection (Run multiple metrics at once)
# ============================================================
class MetricCollection:
    """
    Utility to compute multiple metrics at once.
    """

    def __init__(self, metrics: dict):
        """
        metrics: dict[name -> Metric instance]
        """
        self.metrics = metrics

    def __call__(self, y_true, y_pred):
        return {name: metric(y_true, y_pred) for name, metric in self.metrics.items()}


## **Modeling**

In [ ]:
X_train_selected = X_train[['Curricular units 2nd sem (approved)', 'Course__9773', 'Curricular units 2nd sem (credited)', 'Debtor', 'Tuition fees up to date', 'Course__9500', 'Course__171', 'Course__9119']]

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=settings.SEED)

models = {
    'DT': DecisionTreeClassifier(random_state=settings.SEED),
}

metrics = MetricCollection({
    'accuracy': Accuracy(),
    'f1_macro': F1Score(average='macro'),
    'f1_micro': F1Score(average='micro'),
    'f1_weighted': F1Score(average='weighted')  
})

all_results = {} 

In [ ]:


for name, model in models.items():
    print(f"Training model: {name}")
    fold_results = []  # store metric dict for each fold

    for fold, (train_idx, val_idx) in enumerate(skf.split(train, train[settings.TARGET])):
        
        # -------------------------
        # Split data
        # -------------------------
        X_train = train.drop(columns=[settings.TARGET]).iloc[train_idx]
        y_train_raw = train[settings.TARGET].iloc[train_idx].values
        y_train = label_encoder.transform(y_train_raw)

        X_val = train.drop(columns=[settings.TARGET]).iloc[val_idx]
        y_val_raw = train[settings.TARGET].iloc[val_idx].values
        y_val = label_encoder.transform(y_val_raw)

        # -------------------------
        # Preprocessing
        # -------------------------
        train_data = preprocessor.fit_transform(X_train)
        val_data = preprocessor.transform(X_val)

        # -------------------------
        # Training
        # -------------------------
        model.fit(train_data, y_train)

        # -------------------------
        # Validation
        # -------------------------
        val_preds = model.predict(val_data)

        # -------------------------
        # Metrics
        # -------------------------
        results = metrics(y_val, val_preds)
        fold_results.append(results)

        print(f"Fold {fold + 1} results: {results}")

    # -------------------------
    # Aggregate across folds
    # -------------------------
    df_results = pd.DataFrame(fold_results)
    model_mean = df_results.mean().to_dict()

    all_results[name] = {
        "per_fold": fold_results,
        "mean": model_mean
    }

    print(f"\nAggregated results for model {name}:")
    print(df_results)
    print("Mean metrics:", model_mean)
    print("=" * 60)


Training model: DT
Fold 1 results: {'accuracy': 0.6666666666666666, 'f1_macro': 0.6037784931551394, 'f1_micro': 0.6666666666666666, 'f1_weighted': 0.6680952221620577}
Fold 2 results: {'accuracy': 0.6510500807754442, 'f1_macro': 0.5945259396468542, 'f1_micro': 0.6510500807754442, 'f1_weighted': 0.6554880736211437}


KeyboardInterrupt: 

# ✅**Final Submission**

In [ ]:
train_df = pd.read_csv(settings.TRAIN_PATH)
test_df  = pd.read_csv(settings.TEST_PATH)

X_train_full = train_df.drop(columns=[settings.TARGET])
y_train_full = label_encoder.transform(train_df[settings.TARGET].values)

X_train_full_proc = preprocessor.fit_transform(X_train_full).astype(float)
X_train_full_proc = X_train_full_proc[['Curricular units 2nd sem (approved)', 'Course__9773', 'Curricular units 2nd sem (credited)', 'Debtor', 'Tuition fees up to date', 'Course__9500', 'Course__171', 'Course__9119']]

model = models['DT']
model.fit(X_train_full_proc, y_train_full)

print("Final model trained.")

X_test_proc = preprocessor.transform(test_df).astype(float)
X_test_proc = X_test_proc[['Curricular units 2nd sem (approved)', 'Course__9773', 'Curricular units 2nd sem (credited)', 'Debtor', 'Tuition fees up to date', 'Course__9500', 'Course__171', 'Course__9119']]

test_preds_int = model.predict(X_test_proc)
test_preds = label_encoder.inverse_transform(test_preds_int)

submission = pd.read_csv(settings.SUBMISSION_PATH)

submission[settings.TARGET] = test_preds
submission.to_csv("sub-5.csv", index=False)

print("\nSubmission saved to sub-5.csv")

Final model trained.

Submission saved to sub-5.csv
